### 1) Data loading and first look

In [4]:
import time
import os

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

print(f"Shape of train: {train.shape}")
print(f"Shape of test: {test.shape}")
print(f'Columns in train: {train.columns.tolist()}')
print(f'Columns in test: {test.columns.tolist()}')
train.head()

Shape of train: (10003, 3)
Shape of test: (3080, 3)
Columns in train: ['text', 'label', 'label_text']
Columns in test: ['text', 'label', 'label_text']


,text,label,label_text
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival


### 2) Quality of data

In [5]:
print(f"Missing values in train:\n{train.isna().sum()}")
print(f"\nMissing values in test:\n{test.isna().sum()}")

print(f"\nDuplicates in train: {train.duplicated().sum()}")
print(f"Duplicates in test: {test.duplicated().sum()}")

print(f"\nCount of test's texts in train: {train['text'].isin(test['text']).sum()}")

Missing values in train:
text          0
label         0
label_text    0
dtype: int64

Missing values in test:
text          0
label         0
label_text    0
dtype: int64

Duplicates in train: 0
Duplicates in test: 0

Count of test's texts in train: 0


### 3) Class distribution

In [6]:
train_classes = set(train["label_text"].unique())
test_classes  = set(test["label_text"].unique())

print(f"Unique classes in train: {len(train_classes)}")
print(f"Unique classes in test: {len(test_classes)}")
print(f"Same set of classes: {train_classes == test_classes}")

counts_train = train["label_text"].value_counts()
print(f"\nDescription of train classes: {counts_train.describe()}")
print(f"\nmin / median / max in class: {counts_train.min()} / {int(counts_train.median())} / {counts_train.max()}")
print("\nTop-5 most popular classes:")
print(counts_train.head())
print("\nTop-5 most rare classes:")
print(counts_train.tail())

counts_test = test['label_text'].value_counts()
print(f"\nDescription of test classes: {counts_test.describe()}")


Unique classes in train: 77
Unique classes in test: 77
Same set of classes: True

Description of train classes: count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: count, dtype: float64

min / median / max in class: 35 / 127 / 187

Top-5 most popular classes:
label_text
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
Name: count, dtype: int64

Top-5 most rare classes:
label_text
lost_or_stolen_card         82
card_swallowed              61
card_acceptance             59
virtual_card_not_working    41
contactless_not_working     35
Name: count, dtype: int64

Description of test classes: count    77.0
mean     40.0
std       0.0
min      40.0
25%  

In [7]:
pd.set_option("display.max_colwidth", None)

train["len_words"] = train["text"].str.split().str.len()
train["len_chars"] = train["text"].str.len()
test["len_words"] = test["text"].str.split().str.len()
test["len_chars"] = test["text"].str.len()

print("=" * 10 + " Request length in words " + "=" * 10)
print(f"Train: \n{train["len_words"].describe()}")
print(f"\nTest: \n{test['len_words'].describe()}\n")

print("=" * 10 + " Request length in chars " + "=" * 10)
print(f"Train: \n{train["len_chars"].describe()}")
print(f"\nTest: \n{test['len_chars'].describe()}\n")

print("="*10 + "Top 5 smallest requests in train" + "="*10)
print(train.nsmallest(5, "len_words")[["text", "label", "label_text"]])

print("="*10 + "Top 5 largest requests in train" + "="*10)
print(train.nlargest(5, "len_words")[["text", "label", "label_text"]])

========== Request length in words ==========
Train: 
count    10003.000000
mean        11.949415
std          7.891577
min          2.000000
25%          7.000000
50%         10.000000
75%         13.000000
max         79.000000
Name: len_words, dtype: float64

Test: 
count    3080.000000
mean       10.952597
std         6.688180
min         2.000000
25%         7.000000
50%         9.000000
75%        12.000000
max        69.000000
Name: len_words, dtype: float64

========== Request length in chars ==========
Train: 
count    10003.000000
mean        59.473758
std         40.867901
min         13.000000
25%         36.000000
50%         47.000000
75%         64.000000
max        433.000000
Name: len_chars, dtype: float64

Test: 
count    3080.000000
mean       54.232468
std        34.658008
min        13.000000
25%        35.000000
50%        45.000000
75%        60.000000
max       368.000000
Name: len_chars, dtype: float64

==========Top 5 smallest requests in train==========
     

Смутило, то что длинные ответы в train похоже на ответы поддержки

In [8]:
train["looks_like_support"] = train["text"].str.lower().str.contains(
    r"\b(?:please make sure|double-check|you must|you need to|note:)\b", regex=True
)
print(f"The share of responses: {train['looks_like_support'].mean():.2%}")


The share of responses: 0.17%


In [9]:
vec_11 = TfidfVectorizer(ngram_range = (1, 1), min_df = 1, max_features = 100000, sublinear_tf = True)
vec_12 = TfidfVectorizer(ngram_range  = (1, 2), min_df = 1, max_features = 100000, sublinear_tf = True)

vec_11.fit(train["text"])
vec_12.fit(train["text"])

print(f"Length of (1,1) vocabulary: {len(vec_11.vocabulary_):,}")
print(f"Length of (1,2) vocabulary: {len(vec_12.vocabulary_):,}")
print(f"Growth of length: x{len(vec_12.vocabulary_)/len(vec_11.vocabulary_):.2f}")

Length of (1,1) vocabulary: 2,320
Length of (1,2) vocabulary: 23,608
Growth of length: x10.18


In [10]:
import random
random.seed(67)

vocab_12 = list(vec_12.vocabulary_.keys())
sample = random.sample(vocab_12, 40)

for w in sample: print(f" {w!r}")

 'an app'
 'support other'
 'months ago'
 'buyer'
 'fraud noticed'
 'that think'
 'transfer charged'
 'atm would'
 'for isn'
 'doesnt'
 'some fast'
 'yesterday topped'
 'of pulling'
 'where are'
 'tranfers from'
 'old enough'
 'money went'
 'from you'
 'about exchanging'
 'stuck since'
 'tried the'
 'are multiple'
 'current exchanges'
 'pending message'
 'returned my'
 'there and'
 'than requested'
 'using mine'
 'rewarded instead'
 'gotten my'
 'my knowledge'
 'direction'
 'eaten by'
 'month will'
 'whats types'
 'understanding why'
 'fail double'
 'close my'
 'home yet'
 'it expires'


In [11]:
X_train = train["text"]
y_train = train["label_text"]
X_test = test["text"]
y_test = test["label_text"]

results = []

for ngram in [(1,1), (1,2)]:
    for cw in [None, 'balanced']:
        t0 = time.time()

        vectorizer = TfidfVectorizer(
            ngram_range = ngram,
            min_df = 1,
            max_features = 100000,
            sublinear_tf = True
        )

        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        classifier = LogisticRegression(
            class_weight = cw,
            max_iter = 2000,
            C = 1
        )

        classifier.fit(X_train_vec, y_train)
        y_pred = classifier.predict(X_test_vec)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average = 'macro')
        dt = time.time() - t0

        results.append({
            "ngram": ngram,
            "class_weight": cw,
            "vocab": len(vectorizer.vocabulary_),
            "acc": round(acc, 4),
            "macro_f1": round(f1, 4),
            "time_sec": round(dt, 1)
        })

print(pd.DataFrame(results))

    ngram class_weight  vocab     acc  macro_f1  time_sec
0  (1, 1)          NaN   2320  0.8782    0.8784       1.5
1  (1, 1)     balanced   2320  0.8851    0.8853       1.5
2  (1, 2)          NaN  23608  0.8571    0.8558       8.8
3  (1, 2)     balanced  23608  0.8724    0.8726       6.6


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipe = Pipeline([("tfidf", TfidfVectorizer(sublinear_tf=True)),
                  ("clf", LogisticRegression(class_weight="balanced"))])

param_grid = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [1, 2],
    "tfidf__norm": ["l2", None],
    "clf__C": [0.3, 1.0, 3.0, 10.0]          
              }

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 67)
grid = GridSearchCV(pipe, param_grid, cv = cv, scoring = "f1_macro", n_jobs = -1, verbose = 1)
grid.fit(X_train, y_train)

print(f"The best parameters: {grid.best_params_}")
print(f"The best cv macro-f1: {grid.best_score_:.4f}")

y_pred = grid.best_estimator_.predict(X_test)
print(f"Test macro-F1: {f1_score(y_test, y_pred, average = "macro"):.4f}")


Fitting 5 folds for each of 32 candidates, totalling 160 fits
The best parameters: {'clf__C': 10.0, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__norm': 'l2'}
The best cv macro-f1: 0.8835
Test macro-F1: 0.9037


In [13]:
os.makedirs("../results", exist_ok = True)

out = test.copy()
out["pred_intent"] = y_pred
out.to_csv("../results/baseline_logreg.csv", index = False)

report = classification_report(y_test, y_pred, digits = 3, zero_division = 0)

with open("../results/baseline_report.txt", "w") as f:
    f.write(report)

print("Saved")

Saved


In [14]:
report_df = pd.DataFrame(classification_report(y_test, y_pred,  output_dict = True, zero_division = 0)).T
report_df = report_df.round(4)
print(report_df.head())

print(f"\nTop 10 worst classes:\n {report_df.sort_values("f1-score").head(10)[["precision", "recall", "f1-score", "support"]]}")

                         precision  recall  f1-score  support
Refund_not_showing_up       0.9250   0.925    0.9250     40.0
activate_my_card            0.9744   0.950    0.9620     40.0
age_limit                   1.0000   1.000    1.0000     40.0
apple_pay_or_google_pay     0.9756   1.000    0.9877     40.0
atm_support                 0.9286   0.975    0.9512     40.0

Top 10 worst classes:
                                          precision  recall  f1-score  support
card_not_working                            0.6735   0.825    0.7416     40.0
balance_not_updated_after_bank_transfer     0.7632   0.725    0.7436     40.0
pending_transfer                            0.7692   0.750    0.7595     40.0
top_up_failed                               0.8049   0.825    0.8148     40.0
top_up_reverted                             0.8250   0.825    0.8250     40.0
transfer_not_received_by_recipient          0.8095   0.850    0.8293     40.0
unable_to_verify_identity                   0.8293   0.850

In [15]:
errors = pd.DataFrame({
    "text": test["text"].values,
    "y_true": y_test.values,
    "y_pred": y_pred
})

errors = errors[errors["y_true"] != errors["y_pred"]]
print(f"Errors: {len(errors)} ({len(errors)/len(test):.1%})")

print("\nTop 10 confusion pairs")
print(errors.groupby(["y_true", "y_pred"]).size().sort_values(ascending = False).head(10))

print(f"\nExamples of class card_not_working: {errors[errors["y_true"] == "card_not_working"][["text", "y_pred"]].head(5)}")

Errors: 297 (9.6%)

Top 10 confusion pairs
y_true                                   y_pred                            
balance_not_updated_after_bank_transfer  transfer_not_received_by_recipient    5
card_arrival                             card_delivery_estimate                4
unable_to_verify_identity                verify_my_identity                    4
pending_top_up                           top_up_failed                         4
why_verify_identity                      unable_to_verify_identity             4
top_up_failed                            top_up_reverted                       4
fiat_currency_support                    exchange_via_app                      4
card_delivery_estimate                   card_arrival                          3
balance_not_updated_after_bank_transfer  pending_transfer                      3
fiat_currency_support                    supported_cards_and_currencies        3
dtype: int64

Examples of class card_not_working:                      

In [18]:
test_sample = test.groupby("label_text").head(5).reset_index(drop = True)
test_sample.to_csv("../data/raw/test_sample.csv", index = False)
print(f"Length of test's sample: {len(test_sample)}")

y_pred_base = grid.best_estimator_.predict(test_sample["text"])

baseline_sample = test_sample.copy()
baseline_sample["pred_intent"] = y_pred_base
baseline_sample = baseline_sample[["text","label", "label_text", "pred_intent"]]
baseline_sample.to_csv("../results/logreg_sample_predictions.csv", index=False)

Length of test's sample: 385


In [19]:
test_sample = test.groupby("label_text").head(1).reset_index(drop = True)
test_sample.to_csv("../data/raw/test_sample_v2.csv", index = False)
print(f"Length of test's sample: {len(test_sample)}")

y_pred_base = grid.best_estimator_.predict(test_sample["text"])

baseline_sample = test_sample.copy()
baseline_sample["pred_intent"] = y_pred_base
baseline_sample = baseline_sample[["text","label", "label_text", "pred_intent"]]
baseline_sample.to_csv("../results/logreg_sample_v2_predictions.csv", index=False)

Length of test's sample: 77
